# Titanic — v3: Evidence-Based Feature Selection

**Version 3.** The v2 experiment showed: deleting rows hurts, and of the four "suspect" columns only `Embarked` was truly near-useless. So v3 keeps the full v1 pipeline with exactly one change — **drop `Embarked`**, keep the weak-but-real features (`Fare`, `HasCabin`), and impute all missing values with medians (Age by title-group median, Fare by class median) so every row is kept.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

def add_features(df):
    df = df.copy()
    df["Title"] = df.Name.str.extract(r",\s*([^.]+)\.")[0].str.strip().replace(
        {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["Title"] = df.Title.where(df.Title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    df["FamilySize"] = df.SibSp + df.Parch + 1
    df["IsAlone"] = (df.FamilySize == 1).astype(int)
    df["HasCabin"] = df.Cabin.notna().astype(int)
    return df

train_fe = add_features(train)
test_fe = add_features(test)

# Median imputation, computed on train only (no leakage)
age_by_title = train_fe.groupby("Title").Age.median()
fare_by_class = train_fe.groupby("Pclass").Fare.median()
for df in (train_fe, test_fe):
    df["Age"] = df.Age.fillna(df.Title.map(age_by_title))
    df["Fare"] = df.Fare.fillna(df.Pclass.map(fare_by_class))

FEATURES = ["Age", "Fare", "FamilySize", "Pclass", "Sex", "Title", "IsAlone", "HasCabin"]  # v1 minus Embarked
X, y = train_fe[FEATURES], train_fe.Survived

pre = ColumnTransformer([
    ("num", StandardScaler(), ["Age", "Fare", "FamilySize"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     ["Pclass", "Sex", "Title", "IsAlone", "HasCabin"]),
])
models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Random forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, m in models.items():
    s = cross_val_score(Pipeline([("pre", pre), ("m", m)]), X, y, cv=cv)
    results[name] = s.mean()
    print(f"{name:22s} {s.mean():.4f} ± {s.std():.4f}")

Logistic regression    0.8283 ± 0.0059


Random forest          0.8339 ± 0.0101


Gradient boosting      0.8350 ± 0.0138


Reference — v1 (with Embarked): LogReg 0.8361, RF 0.8350, GB 0.8440. If v3 matches v1 within noise, `Embarked` confirmed as dead weight: same accuracy, one less feature — the simpler model wins on principle (Occam).

## v3 submission

In [2]:
best = max(results, key=results.get)
print("Best model:", best, f"({results[best]:.4f})")

final = Pipeline([("pre", pre), ("m", models[best])])
final.fit(X, y)
preds = final.predict(test_fe[FEATURES])

sub = pd.DataFrame({"PassengerId": test_fe.PassengerId, "Survived": preds})
sub.to_csv("submission_v3.csv", index=False)
print(f"submission_v3.csv written — predicted survival rate: {preds.mean():.3f}")

Best model: Gradient boosting (0.8350)


submission_v3.csv written — predicted survival rate: 0.366


## Version history

| Version | Change | Best CV | Leaderboard |
|---|---|---|---|
| v1 | Full pipeline, all features | 0.844 (GB) | 0.76794 |
| v2 | Lean features + missing-Age rows deleted | 0.824 (LR) | 0.75837 |
| v3 | v1 minus Embarked, median imputation, all rows kept | (above) | — |
